# pFedMe

> The implementation of pFedMe server as in https://arxiv.org/pdf/2006.08848

In [ ]:
#| default_exp servers.pfedme

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import os

from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fastcore.utils import *
from fastcore.all import *

import torch
import torch.nn as nn
import torch.nn.functional as F

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## pFedMe  
Personalized Federated learning using Morseu-envelope

In [ ]:
#| export
@AlgorithmRegistry.register_server("pfedme")
class ServerpFedMe(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
        

### Aggregation

In [ ]:
#| export
@patch
def aggregate(self: ServerpFedMe, lst_active_ids, comm_round, len_clients_ds):

    m_t = sum(len_clients_ds.values())
    with torch.no_grad():

        prev_global_model = self.model.state_dict()
        for i, id in enumerate(lst_active_ids):
            # state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
            # state = torch.load(state_path, weights_only=False)
            state = self.state_mgr.get_state(id)
            client_state_dict = state['model']

            if i == 0:
                global_model = {
                    key: torch.zeros_like(value) 
                    for key, value in client_state_dict.items()
                }

            n_k = len_clients_ds[id]
            weight =  n_k / m_t 

            for key in client_state_dict.keys():
                global_model[key].add_(weight * client_state_dict[key])

        for key in global_model.keys():
            global_model[key].copy_((1-self.cfg.algorithm.beta)*global_model[key] + self.cfg.algorithm.beta*prev_global_model[key])

        self.model.load_state_dict(global_model)
        # server_state = {
        #     'model': global_model,
        # }
        # server_state_path = os.path.join(self.cfg.save_dir, str(comm_round), "global_model", "state.pth")
        # if not os.path.exists(os.path.dirname(server_state_path)):
        #     os.makedirs(os.path.dirname(server_state_path), exist_ok=True)
            
        # torch.save(server_state, server_state_path)

    for i, id in enumerate(lst_active_ids):
        client_state = self.state_mgr.get_state(id)
        client_state['model'] = global_model
        self.state_mgr.set_state(id, client_state)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()